<a href="https://colab.research.google.com/github/Santiago-Soria/proyecto-transformacion-texto-imagen/blob/sant_branch/notebooks/3.2_run_experimentos_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Clonar repositorio
!git clone https://github.com/Santiago-Soria/proyecto-transformacion-texto-imagen.git


fatal: destination path 'proyecto-transformacion-texto-imagen' already exists and is not an empty directory.


In [2]:
# Cambiar al directorio del proyecto
%cd proyecto-transformacion-texto-imagen

/content/proyecto-transformacion-texto-imagen


In [4]:
!git fetch
!git checkout sant_branch
!git pull origin sant_branch

M	src/models/train_classic.py
Already on 'sant_branch'
Your branch is behind 'origin/sant_branch' by 3 commits, and can be fast-forwarded.
  (use "git pull" to update your local branch)
From https://github.com/Santiago-Soria/proyecto-transformacion-texto-imagen
 * branch            sant_branch -> FETCH_HEAD
Updating 9fa6a61..cf96e55
error: Your local changes to the following files would be overwritten by merge:
	src/models/train_classic.py
Please commit your changes or stash them before you merge.
Aborting


In [5]:
!git branch

  main
* sant_branch


In [6]:
# Instalar dependencias del proyecto
!pip install transformers datasets torch scikit-learn pandas numpy spacy importlib matplotlib-venn
!python -m spacy download es_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 74.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [7]:
import torch
print(f"GPU disponible: {torch.cuda.is_available()}")
print(f"Dispositivo: {torch.cuda.get_device_name(0)}")

GPU disponible: True
Dispositivo: Tesla T4


In [8]:
import sys
import os
import importlib
# Agregar src al path desde la raíz del proyecto
sys.path.insert(0, '/content/proyecto-transformacion-texto-imagen/src')

from models import train_classic
# Recargar el módulo
importlib.reload(train_classic)

# Ahora importar normalmente
import pandas as pd
from preprocessing_utils import preprocess_text
from features.extraction import FeatureExtractor
from models.train_classic import train_logistic
from models.train_transformer import run_finetuning

In [9]:
from re import X
train = pd.read_csv('/content/proyecto-transformacion-texto-imagen/data/processed/train.csv')
test = pd.read_csv('/content/proyecto-transformacion-texto-imagen/data/processed/test.csv')

# Limpieza Mínima en el Preprocesamiento
X_train_p1 = [preprocess_text(t, 'P1') for t in train['text']]
X_test_p1 = [preprocess_text(t, 'P1') for t in test['text']]

# Extractor de Caractetísticas
extractor =  FeatureExtractor()

### **Experimento 1.1:** Limpieza Básica + TF-IDF + Regresión Logística

In [12]:
# Vectorizamos con TF-IDF
X_train_tfidf, vectorizer = extractor.get_tfidf(X_train_p1)
X_test_tfidf = vectorizer.transform(X_test_p1)

# Entrenamos con Regresión Logística
train_logistic(X_train_tfidf, train['manual_classification'],
               X_test_tfidf, test['manual_classification'], "Exp_1.1_TFIDF_Baseline")


Entrenando Regresión Logística para: Exp_1.1_TFIDF_Baseline

Resultados:
Accuracy: 0.7018
Precision (Macro): 0.6854
Recall (Macro): 0.6854
F1 (Macro): 0.6854

Confusion Matrix:
[[53 17]
 [17 27]]

📄 Classification Report:
              precision    recall  f1-score   support

           0       0.76      0.76      0.76        70
           1       0.61      0.61      0.61        44

    accuracy                           0.70       114
   macro avg       0.69      0.69      0.69       114
weighted avg       0.70      0.70      0.70       114


Modelo guardado en: /content/proyecto-transformacion-texto-imagen/src/models/../../models/Exp_1.1_TFIDF_Baseline.pkl
Métricas guardadas en: /content/proyecto-transformacion-texto-imagen/src/models/../../models/../results/Exp_1.1_TFIDF_Baseline_metrics.json


(LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42,
                    solver='liblinear'),
 array([1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0,
        1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1,
        1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0,
        0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0,
        1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0,
        1, 0, 1, 0]),
 {'accuracy': 0.7017543859649122,
  'precision_macro': 0.6853896103896104,
  'recall_macro': 0.6853896103896104,
  'f1_macro': 0.6853896103896104})

### **Experimento 1.2:** Limpieza Básica + Stopwords + Lematización + TF-IDF + Regresión Logística

In [13]:
# Usamos P2 (Quitamos stopwords)
X_train_p2 = [preprocess_text(t, 'P2') for t in train['text']]
X_test_p2 = [preprocess_text(t, 'P2') for t in test['text']]

# Vectorizamos con TF-IDF
X_train_tfidf_sw, vec_sw = extractor.get_tfidf(X_train_p2)
X_test_tfidf_sw = vec_sw.transform(X_test_p2)

# Entrenamos con Regresión Logística
train_logistic(X_train_tfidf_sw, train['manual_classification'],
               X_test_tfidf_sw, test['manual_classification'], "Exp_1.2_TFIDF_Stopwords")


Entrenando Regresión Logística para: Exp_1.2_TFIDF_Stopwords

Resultados:
Accuracy: 0.7018
Precision (Macro): 0.6867
Recall (Macro): 0.6896
F1 (Macro): 0.6879

Confusion Matrix:
[[52 18]
 [16 28]]

📄 Classification Report:
              precision    recall  f1-score   support

           0       0.76      0.74      0.75        70
           1       0.61      0.64      0.62        44

    accuracy                           0.70       114
   macro avg       0.69      0.69      0.69       114
weighted avg       0.70      0.70      0.70       114


Modelo guardado en: /content/proyecto-transformacion-texto-imagen/src/models/../../models/Exp_1.2_TFIDF_Stopwords.pkl
Métricas guardadas en: /content/proyecto-transformacion-texto-imagen/src/models/../../models/../results/Exp_1.2_TFIDF_Stopwords_metrics.json


(LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42,
                    solver='liblinear'),
 array([1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0,
        1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1,
        1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0,
        0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0,
        1, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0,
        1, 0, 1, 1]),
 {'accuracy': 0.7017543859649122,
  'precision_macro': 0.6867007672634271,
  'recall_macro': 0.6896103896103896,
  'f1_macro': 0.6879227053140097})

### **Experimento 2.2:** Limpieza Básica + RoBERTa-BNE (Frozen) + Regresión Logística

In [11]:
# ==========================================
# EXP 2.2: Frozen RoBERTa + LR
# ==========================================
# Usamos P1 (Transformers prefieren texto con estructura, no P2)
print("Generando embeddings. . .")
X_train_emb = extractor.get_transformer_embeddings(X_train_p1, "bertin-project/bertin-roberta-base-spanish")
X_test_emb = extractor.get_transformer_embeddings(X_test_p1, "bertin-project/bertin-roberta-base-spanish")

models_dir = '/content/proyecto-transformacion-texto-imagen/models'
os.makedirs(models_dir, exist_ok=True)

train_logistic(X_train_emb, train['manual_classification'],
               X_test_emb, test['manual_classification'], "Exp_2.2_RoBERTa_Frozen")

Generando embeddings. . .
--> Extrayendo embeddings con bertin-project/bertin-roberta-base-spanish...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: bertin-project/bertin-roberta-base-spanish
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
100%|██████████| 29/29 [00:06<00:00,  4.70it/s]


--> Extrayendo embeddings con bertin-project/bertin-roberta-base-spanish...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: bertin-project/bertin-roberta-base-spanish
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
100%|██████████| 4/4 [00:00<00:00,  5.30it/s]



Entrenando Regresión Logística para: Exp_2.2_RoBERTa_Frozen

Resultados:
Accuracy: 0.6228
Precision (Macro): 0.6089
Recall (Macro): 0.6127
F1 (Macro): 0.6096

Confusion Matrix:
[[46 24]
 [19 25]]

📄 Classification Report:
              precision    recall  f1-score   support

           0       0.71      0.66      0.68        70
           1       0.51      0.57      0.54        44

    accuracy                           0.62       114
   macro avg       0.61      0.61      0.61       114
weighted avg       0.63      0.62      0.63       114


Modelo guardado en: /content/proyecto-transformacion-texto-imagen/src/models/../../models/Exp_2.2_RoBERTa_Frozen.pkl
Métricas guardadas en: /content/proyecto-transformacion-texto-imagen/src/models/../../models/../results/Exp_2.2_RoBERTa_Frozen_metrics.json


(LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42,
                    solver='liblinear'),
 array([0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0,
        1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 1,
        1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0,
        0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0,
        1, 0, 1, 1]),
 {'accuracy': 0.6228070175438597,
  'precision_macro': 0.6089481946624804,
  'recall_macro': 0.6126623376623377,
  'f1_macro': 0.609557945041816})

### **Experimento 2.3:** Limpieza Básica + mDeBERTa-v3-base (Frozen) + Regresión Logística

In [14]:
# ==========================================
# EXP 2.3: Frozen mDeBERTa-v3-base + LR
# ==========================================
# Usamos P1 (Transformers prefieren texto con estructura, no P2)
print("Generando embeddings. . .")
X_train_emb = extractor.get_transformer_embeddings(X_train_p1, "microsoft/mdeberta-v3-base")
X_test_emb = extractor.get_transformer_embeddings(X_test_p1, "microsoft/mdeberta-v3-base")

models_dir = '/content/proyecto-transformacion-texto-imagen/models'
os.makedirs(models_dir, exist_ok=True)

train_logistic(X_train_emb, train['manual_classification'],
               X_test_emb, test['manual_classification'], "Exp_2.3_mDeBERTa_Frozen")

Generando embeddings. . .
--> Extrayendo embeddings con microsoft/mdeberta-v3-base...


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

...

--> Extrayendo embeddings con microsoft/mdeberta-v3-base...


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/mdeberta-v3-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
lm_predictions.lm_head.bias                | UNEXPECTED |  | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight          | UNEXPECTED |  | 
mask_predictions.dense.weight              | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED |  | 
mask_predictions.dense.bias                | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias            | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED |  | 
mask_predictions.classifier.weight         | UNEXPECTED |  | 
mask_predictions.classifier.bias           | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/ar


Entrenando Regresión Logística para: Exp_2.3_mDeBERTa_Frozen

Resultados:
Accuracy: 0.5526
Precision (Macro): 0.5262
Recall (Macro): 0.5260
F1 (Macro): 0.5260

Confusion Matrix:
[[45 25]
 [26 18]]

📄 Classification Report:
              precision    recall  f1-score   support

           0       0.63      0.64      0.64        70
           1       0.42      0.41      0.41        44

    accuracy                           0.55       114
   macro avg       0.53      0.53      0.53       114
weighted avg       0.55      0.55      0.55       114


Modelo guardado en: /content/proyecto-transformacion-texto-imagen/src/models/../../models/Exp_2.3_mDeBERTa_Frozen.pkl
Métricas guardadas en: /content/proyecto-transformacion-texto-imagen/src/models/../../models/../results/Exp_2.3_mDeBERTa_Frozen_metrics.json


(LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42,
                    solver='liblinear'),
 array([1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,
        1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1,
        0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0,
        0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0,
        1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1,
        1, 0, 0, 1]),
 {'accuracy': 0.5526315789473685,
  'precision_macro': 0.5262037340320996,
  'recall_macro': 0.525974025974026,
  'f1_macro': 0.5260454878943507})

### **Experimento 3.2:** Limpieza Básica + RoBERTa-BNE(Fine-Tuning) + Softmax

In [17]:
# ==========================================
# EXP 3.2: Fine-Tuning RoBERTa
# ==========================================
trainer = run_finetuning(X_train_p1, train['manual_classification'].values,
                         X_test_p1, test['manual_classification'].values,
                         "bertin-project/bertin-roberta-base-spanish")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: bertin-project/bertin-roberta-base-spanish
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Precision Macro,Recall Macro,F1 Macro
1,0.670759,0.653055,0.614035,0.307018,0.500000,0.380435
2,0.607349,0.531289,0.710526,0.730415,0.646104,0.644927
3,0.519264,0.700945,0.710526,0.700155,0.667208,0.672157
4,0.381932,0.845541,0.684211,0.677778,0.624675,0.622794
5,0.193311,1.059833,0.710526,0.697531,0.671429,0.676443
6,0.117746,1.187839,0.719298,0.717608,0.670130,0.675214


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


Final Evaluation Metrics:
{'eval_loss': 1.0598838329315186, 'eval_accuracy': 0.7105263157894737, 'eval_precision_macro': 0.6975308641975309, 'eval_recall_macro': 0.6714285714285715, 'eval_f1_macro': 0.6764427625354777, 'eval_runtime': 0.9605, 'eval_samples_per_second': 118.687, 'eval_steps_per_second': 8.329, 'epoch': 6.0}

Confusion Matrix:
[[59 11]
 [22 22]]


### **Experimento 3.3:** Limpieza Básica + mDeBERTa-v3-base + Softmax

In [19]:
# ==========================================
# EXP 3.33: Fine-Tuning RoBERTa
# ==========================================
# Este es el pesado. Asegúrate de estar en GPU.
trainer = run_finetuning(X_train_p1, train['manual_classification'].values,
                         X_test_p1, test['manual_classification'].values,
                         "microsoft/mdeberta-v3-base")

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/mdeberta-v3-base
Key                                        | Status     | 
-------------------------------------------+------------+-
lm_predictions.lm_head.bias                | UNEXPECTED | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED | 
mask_predictions.LayerNorm.weight          | UNEXPECTED | 
mask_predictions.dense.weight              | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED | 
mask_predictions.dense.bias                | UNEXPECTED | 
mask_predictions.LayerNorm.bias            | UNEXPECTED | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED | 
mask_predictions.classifier.weight         | UNEXPECTED | 
mask_predictions.classifier.bias           | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
classifier.bias                    

Epoch,Training Loss,Validation Loss,Accuracy,Precision Macro,Recall Macro,F1 Macro
1,1.120104,nan,0.614035,0.307018,0.500000,0.380435
2,0.000000,nan,0.614035,0.307018,0.500000,0.380435
3,0.000000,nan,0.614035,0.307018,0.500000,0.380435


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['deberta.embeddings.LayerNorm.weight', 'deberta.embeddings.LayerNorm.bias', 'deberta.encoder.layer.0.attention.output.LayerNorm.weight', 'deberta.encoder.layer.0.attention.output.LayerNorm.bias', 'deberta.encoder.layer.0.output.LayerNorm.weight', 'deberta.encoder.layer.0.output.LayerNorm.bias', 'deberta.encoder.layer.1.attention.output.LayerNorm.weight', 'deberta.encoder.layer.1.attention.output.LayerNorm.bias', 'deberta.encoder.layer.1.output.LayerNorm.weight', 'deberta.encoder.layer.1.output.LayerNorm.bias', 'deberta.encoder.layer.2.attention.output.LayerNorm.weight', 'deberta.encoder.layer.2.attention.output.LayerNorm.bias', 'deberta.encoder.layer.2.output.LayerNorm.weight', 'deberta.encoder.layer.2.output.LayerNorm.bias', 'deberta.encoder.layer.3.attention.output.LayerNorm.weight', 'deberta.encoder.layer.3.attention.output.LayerNorm.bias', 'deberta.encoder.layer.3.output.LayerNorm.weight', 'deberta.encoder.layer.3.output.Laye


Final Evaluation Metrics:
{'eval_loss': nan, 'eval_accuracy': 0.6140350877192983, 'eval_precision_macro': 0.30701754385964913, 'eval_recall_macro': 0.5, 'eval_f1_macro': 0.3804347826086957, 'eval_runtime': 0.5721, 'eval_samples_per_second': 199.251, 'eval_steps_per_second': 13.983, 'epoch': 3.0}


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



Confusion Matrix:
[[70  0]
 [44  0]]
